# レッスン 10 - 本番環境での AI エージェント

In this lesson you will learn **本番運用パターン** for AI agents using the Microsoft Agent Framework (Python). We cover:

- **可観測性** — エージェントのやり取りにタイミングとログを追加する
- **評価** — 評価エージェントを使用してレスポンスの品質をスコアリングする
- **コスト管理** — トークン最適化とモデル選択の戦略

The scenario is a **旅行代理店** that helps users plan trips, with monitoring and evaluation layered on top.


## セットアップ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## 本番環境での考慮事項

AIエージェントをプロトタイプから本番環境へ移行するには、次の3つの柱に慎重な注意が必要です:

1. **可観測性** — エージェントが何をしているか、どれくらい時間がかかっているか、どのツールを呼び出しているかを可視化する必要があります。トレースとログがなければ、本番環境の問題をデバッグすることはほとんど不可能です。

2. **評価** — 自動化された品質チェックは、エージェントの応答が時間の経過とともに正確で、完全で、有用なままであることを保証します。評価エージェントは定義された基準に沿って応答をスコア化できます。

3. **コスト管理** — トークン使用量はコストに直接影響します。プロンプトの最適化、モデルの選択、キャッシュなどの戦略は、品質を犠牲にせずに費用を抑えるのに役立ちます。


## 観測可能なエージェントの構築

旅行ツールを定義し、エージェントの呼び出しをタイミング計測でラップしてレイテンシを監視できるようにします。本番環境では OpenTelemetry や同様のトレーシングバックエンドと統合します。


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## 評価パターン

一般的な本番環境でのパターンは、2つ目のエージェントを**評価者**として使用することです。評価者は、一次エージェントの応答を、完全性、正確性、有用性などの事前定義された基準に照らして採点します。

これにより:
- ユーザーに届く前の自動化された品質ゲート
- プロンプトやモデルが変更されたときの回帰検出
- エージェントのパフォーマンスを継続的に監視


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## コスト管理戦略

コスト管理は本番環境のAIエージェントにとって重要です。ここでは主要な戦略を紹介します:

| 戦略 | 説明 |
|---|---|
| **プロンプト最適化** | システム指示を簡潔に保ちます。冗長なコンテキストを削除して入力トークンを減らします。 |
| **モデル選択** | 分類や抽出のような簡単なタスクには小型で安価なモデル（例: GPT-4o-mini）を使用し、複雑な推論にはより大きなモデルを割り当てます。 |
| **キャッシュ** | ツールの結果や頻繁なクエリをキャッシュして、不要なAPIコールを避けます。 |
| **トークン予算** | `max_tokens` の上限を設定して予期せず長くなる応答を防ぎます。 |
| **バッチ処理** | 可能な場合は複数のユーザーのクエリを単一のAPIコールにまとめます。 |

実際には、階層化したアプローチが有効です: 単純なリクエストは高速で安価なモデルにルーティングし、複雑なクエリだけをより高性能なモデルにエスカレーションします。


## 概要

このレッスンでは以下のことを学びました:

1. **可観測性を追加する** エージェントのやり取りにタイミングとログ記録を導入し、トレーシングとモニタリングの基盤を築く。
2. **エージェントの応答を評価する** 完全性、正確性、有用性に基づいてスコアを付ける評価者エージェントを使用して自動的に評価する。
3. **コストを管理する** プロンプトの最適化、モデルの選択、キャッシュ、トークン予算を通じてコストを管理する。

これらの本番運用パターンは、AIエージェントが大規模でも信頼性が高く、測定可能で、コスト効率に優れていることを確保するのに役立ちます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責事項：
この文書は AI 翻訳サービス「Co-op Translator」(https://github.com/Azure/co-op-translator) を使用して翻訳されました。正確性には努めていますが、自動翻訳には誤りや不正確な箇所が含まれる可能性があることをご承知ください。原文（原言語の文書）を正式な情報源としてご参照ください。重要な内容については、プロの翻訳者による翻訳をお勧めします。本翻訳の利用により生じたいかなる誤解や誤訳についても、当社は責任を負いません。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
